In [1]:
%cd E:\SRP\SRP-2025-Project
import os,sys
notebook_dir = os.getcwd()
path = os.path.abspath(os.path.join(notebook_dir, "Code/CMC Model"))
sys.path.append(path)
import torch
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score,precision_recall_curve,roc_curve
from generateSplits import generateSplits
from Dataset import ModelDataset
from model2 import Model
from torch.utils.data import DataLoader
from torch import nn
from copy import deepcopy
from trainModel import evaluate_roc_auc
import numpy as np
import matplotlib.pyplot as plt

E:\SRP\SRP-2025-Project


In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

metadata = pd.read_csv("../Datasets/BreastDCEDL_spy1/BreastDCEDL_spy1_metadata.csv")
train_df,val_df = generateSplits(metadata,0.2,seed=42)
train_df = train_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
val_df = val_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=42)
backup_path = "study_backup.db"

In [3]:
best_params = {'lr': 0.010203264127576701, 'weight_decay': 0.004617603477095672, 'batch_size': 8, 'optimiser_name': 'Adam'}

In [4]:
def get_scores(model,val_loader):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for images,mols,labels in val_loader:
            images=images.to(device)
            mols = mols.to(device)
            labels=labels.to(device)
            logits = model(images,mols)
            score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return y_true,y_score

In [5]:
def trainModel(model:nn.Module, train_loader,optimiser=None,device=torch.device("cpu"),num_epochs=10,val_loader=None,patience=5,scheduler=None):
    model = model.to(device)
    loss_fn = torch.nn.CrossEntropyLoss()
    best_average_precision = 0
    stale_epochs = 0
    best_model_state_dict = None
    if optimiser is None:
        optimiser = torch.optim.AdamW(model.parameters(),lr=1e-3)
    
    for epoch in range(num_epochs):
        model.train()
        losses = torch.tensor(0.0, device=device)
        for images, mol, labels in train_loader:
            images = images.to(device)
            mol = mol.to(device)
            labels = labels.to(device)
            optimiser.zero_grad()
            logits = model(images,mol)
            loss:torch.Tensor = loss_fn(logits,labels)
            loss.backward()
            optimiser.step()
            losses+=loss.detach()
        avg_loss = (losses/len(train_loader)).item()
        print(f"Epoch {epoch} Done. Avg Loss: {avg_loss:.4f}")
        
        if val_loader is not None:
            y_true,y_scores = get_scores(model,val_loader)
            average_precision = average_precision_score(y_true,y_scores)
            print(f"Average Precision: {average_precision}")
            
            if average_precision > best_average_precision:
                best_average_precision = average_precision
                stale_epochs=0
                best_model_state_dict=deepcopy(model.state_dict())
            else:
                stale_epochs+=1
                if stale_epochs>=patience:
                    print(f"Early stopping triggered at epoch {epoch} (best AP: {best_average_precision:.4f}).")
                    break
            if scheduler is not None:
                old_lrs = [group['lr'] for group in optimiser.param_groups]
                if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(average_precision)
                else:
                    scheduler.step()
                new_lrs = [group['lr'] for group in optimiser.param_groups]
                if old_lrs != new_lrs:
                    print(f"Learning rate changed from {old_lrs} to {new_lrs}")
    if best_model_state_dict is not None:
        model.load_state_dict(best_model_state_dict)
            
    return model,best_average_precision

In [6]:
def four_fold_cv_train(params,class_samples,num_epochs):
    lr = params["lr"]
    weight_decay = params["weight_decay"]
    batch_size = params["batch_size"] 
    
    
    roc_auc_scores = []
    average_precision_scores = []
    for train_index, val_index in skf.split(train_df,train_df["pCR"]):
        model = Model()
        model.to(device)
        optimiser = torch.optim.Adam(model.parameters(),lr=lr,weight_decay=weight_decay)
        fold_train_df = train_df.iloc[train_index]
        fold_val_df = train_df.iloc[val_index]
        fold_train_dataset = ModelDataset(fold_train_df,class_samples=class_samples,loading_bar=False,caching=True)
        fold_train_loader = DataLoader(fold_train_dataset,batch_size=batch_size,shuffle=True)
        fold_val_dataset = ModelDataset(fold_val_df,class_samples={0:1,1:1},loading_bar=False,caching=True)
        fold_val_loader = DataLoader(fold_val_dataset,batch_size=batch_size)
        model,score = trainModel(model,fold_train_loader,optimiser,device=device,num_epochs=num_epochs,val_loader=fold_val_loader,patience=num_epochs//4)
        average_precision_scores.append(score)
        roc_auc_scores.append(evaluate_roc_auc(model,fold_val_loader))
    return sum(roc_auc_scores)/len(roc_auc_scores),sum(average_precision_scores)/len(average_precision_scores)

In [7]:
roc_auc_score, averagePrecisionScore = four_fold_cv_train(best_params,class_samples={0:3,1:8},num_epochs=20)
print(roc_auc_score, averagePrecisionScore)

E:\SRP\env\Lib\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Dataset initialised with 409 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 5.6449
Average Precision: 0.2663905388090171
Epoch 1 Done. Avg Loss: 0.7050
Average Precision: 0.3970388986013986
Epoch 2 Done. Avg Loss: 0.6740
Average Precision: 0.37084765576613404
Epoch 3 Done. Avg Loss: 0.6742
Average Precision: 0.38580627705627707
Epoch 4 Done. Avg Loss: 0.6754
Average Precision: 0.3526300465838509
Epoch 5 Done. Avg Loss: 0.6672
Average Precision: 0.5194756054131053
Epoch 6 Done. Avg Loss: 0.6813
Average Precision: 0.4096030517470146
Epoch 7 Done. Avg Loss: 0.6682
Average Precision: 0.42567432567432567
Epoch 8 Done. Avg Loss: 0.6679
Average Precision: 0.5463078965117008
Epoch 9 Done. Avg Loss: 0.6499
Average Precision: 0.48934371549503125
Epoch 10 Done. Avg Loss: 0.7025
Average Precision: 0.48976162726162725
Epoch 11 Done. Avg Loss: 0.6667
Average Precision: 0.39024621212121213
Epoch 12 Done. Avg Loss: 0.6670
Average Precision: 0.46638363571413105
Epoch 13 Done. Avg

E:\SRP\env\Lib\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Dataset initialised with 409 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 6.6250
Average Precision: 0.21789823725189
Epoch 1 Done. Avg Loss: 0.8558
Average Precision: 0.45868298368298366
Epoch 2 Done. Avg Loss: 0.7034
Average Precision: 0.5682040998217468
Epoch 3 Done. Avg Loss: 0.6844
Average Precision: 0.36368394278001054
Epoch 4 Done. Avg Loss: 0.6794
Average Precision: 0.4692708333333333
Epoch 5 Done. Avg Loss: 0.6553
Average Precision: 0.5005700987608883
Epoch 6 Done. Avg Loss: 0.7083
Average Precision: 0.3926656869961824
Epoch 7 Done. Avg Loss: 0.6520
Average Precision: 0.42180788982259565
Early stopping triggered at epoch 7 (best AP: 0.5682).


E:\SRP\env\Lib\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Dataset initialised with 404 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 4.8701
Average Precision: 0.40599543825350287
Epoch 1 Done. Avg Loss: 0.6492
Average Precision: 0.4847322451367627
Epoch 2 Done. Avg Loss: 0.6528
Average Precision: 0.4952281922870158
Epoch 3 Done. Avg Loss: 0.6345
Average Precision: 0.3641375684544565
Epoch 4 Done. Avg Loss: 0.6456
Average Precision: 0.573872622797354
Epoch 5 Done. Avg Loss: 0.6758
Average Precision: 0.46421774423286827
Epoch 6 Done. Avg Loss: 0.6277
Average Precision: 0.4948924920255921
Epoch 7 Done. Avg Loss: 0.6618
Average Precision: 0.30481808152860784
Epoch 8 Done. Avg Loss: 0.6400
Average Precision: 0.35020089475624866
Epoch 9 Done. Avg Loss: 0.6311
Average Precision: 0.4561228531816767
Early stopping triggered at epoch 9 (best AP: 0.5739).


E:\SRP\env\Lib\site-packages\torch\nn\init.py:582: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Dataset initialised with 404 entries.
Dataset initialised with 31 entries.
Epoch 0 Done. Avg Loss: 6.1862
Average Precision: 0.2585183937674359
Epoch 1 Done. Avg Loss: 0.7422
Average Precision: 0.36572055395584807
Epoch 2 Done. Avg Loss: 0.6714
Average Precision: 0.311191135843941
Epoch 3 Done. Avg Loss: 0.7116
Average Precision: 0.3685908227574894
Epoch 4 Done. Avg Loss: 0.6761
Average Precision: 0.4205192955192955
Epoch 5 Done. Avg Loss: 0.6679
Average Precision: 0.3285538936467729
Epoch 6 Done. Avg Loss: 0.6563
Average Precision: 0.39928973675103707
Epoch 7 Done. Avg Loss: 0.6580
Average Precision: 0.5310316616456967
Epoch 8 Done. Avg Loss: 0.6618
Average Precision: 0.6443562610229276
Epoch 9 Done. Avg Loss: 0.6493
Average Precision: 0.47862274601405036
Epoch 10 Done. Avg Loss: 0.6338
Average Precision: 0.5300522089995774
Epoch 11 Done. Avg Loss: 0.6499
Average Precision: 0.5507020757020757
Epoch 12 Done. Avg Loss: 0.6351
Average Precision: 0.3644119769119769
Epoch 13 Done. Avg Loss